### Using KnowRob in Python



This notebook demonstrates how to use the KnowRob system directly in Python. It includes importing necessary modules, initializing the knowledge base, and executing queries.

### Importing KnowRob Modules

In [ ]:
import json

import owlready2
from knowrob import *

First, we import the required modules from KnowRob. The `try-except` block ensures compatibility with different ROS environments, either using the ROS1-specific package or directly loading `knowrob.so`.

In [ ]:
InitKnowRob()

The `InitKnowRob()` function initializes the KnowRob system, setting up necessary configurations and connections.

### Setting Up Knowledge Base

In [ ]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			{"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"}
            # {"alias": "pizza", "uri": "http://www.co-ode.org/ontologies/pizza/pizza.owl"}
		]
	},
	"data-sources": [
		{"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
        # {"path": "tests/owl/pizza.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "swrl2",
			"read-only": False
		}
	],
	"reasoner": [
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)

This block defines the configuration for the KnowledgeBase, including logging, semantic web prefixes, data sources, and backends. The configuration is then serialized to a JSON string and used to initialize the `KnowledgeBase` instance.

### Submitting a Query

In [ ]:
phi1 = QueryParser.parse("swrl_test:hasAncestor(swrl_test:'Fred', ?y)")
# phi1 = QueryParser.parse("pizza:hasCountryOfOrigin(pizza:'AmericanSlicer', ?y)")
# phi1 = QueryParser.parse("pizza:hasCountryOfOrigin(pizza:'Mozarella', ?y)")

Here, a query is parsed using the `QueryParser`. The query checks for ancestors of the entity `Lea` within the `swrl_test` namespace.

### Retrieving Query Results

In [ ]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

The query formulated in the previous step is submitted to the KnowledgeBase. The results are retrieved as a stream, and a queue is created to handle them.

### Processing Query Results


In [ ]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

This block checks if the result is affirmative (`AnswerYes`) and prints each substitution found in the query result, listing variable bindings.

### Negative Query Result Handling


In [ ]:
phi2 = QueryParser.parse("swrl_test:hasSibling(swrl_test:'Ernest', swrl_test:'Fred')")
resultStream = kb.submitQuery(phi2, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult2 = resultQueue.pop_front()
if isinstance(nextResult2, AnswerNo):
    print("result is negative")
else:
    print("result is positive")

A second query checks for a specific condition, in this case, whether `Lea` is an ancestor of herself, which is expected to be false. The result is handled accordingly.

### Inconclusive Query Result Handling

In [ ]:
phi3 = QueryParser.parse("r(?x, ?y)")
resultStream = kb.submitQuery(phi3, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult3 = resultQueue.pop_front()
if isinstance(nextResult3, AnswerDontKnow):
    print("We can't say if the result is true or false")


The final example demonstrates handling a situation where the system cannot determine the truth value of the query, resulting in an `AnswerDontKnow` response.

### Ontology Reasoner

In [ ]:
import json
from knowrob import *
InitKnowRob()

In [ ]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			{"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"},
            # {"alias": "pizza", "uri": "http://www.co-ode.org/ontologies/pizza/pizza.owl"}
            {"alias": "nlquery", "uri": "http://knowrob.org/kb/nlquery"}
		]
	},
	"data-sources": [
		{"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
        # {"path": "tests/owl/pizza.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "swrl",
			"read-only": False
		}
	],
	"reasoner": [
        {
            "name": "OntoQueryReasoner",
            "type": "OntoQueryReasoner",
            "module": "/home/malineni/ROS_WS/knowrob/tutorials/OntoReasoner.py",
			"data-backend": "mongodb",
        }
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)

In [ ]:
phi1 = QueryParser.parse('nlquery:nlquery("get the child classes of the Locomotion class", ?response)')
phi1

In [ ]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

In [ ]:
type(nextResult1)

In [ ]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

In [ ]:
for bind in nextResult1.substitution():
    variable = bind[1]
    term = bind[2]

In [ ]:
print()

### ActionDesignator Reasoner

In [1]:
import json
from knowrob import *
InitKnowRob()

[15:40:31.920] [info] [KnowRob] static initialization done.


In [2]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			# {"alias": "swrl_test", "uri": "http://knowrob.org/kb/swrl_test"},
            {"alias": "soma", "uri": "http://www.ease-crc.org/ont/"},
            {"alias": "dul", "uri": "http://www.ontologydesignpatterns.org/ont/dul/"},
            {"alias": "nlquery", "uri": "http://knowrob.org/kb/nlquery"}
		]
	},
	"data-sources": [
		# {"path": "tests/owl/swrl.owl", "format": "rdf-xml"}
        {"path": "tests/owl/SOMA.owl", "format": "rdf-xml"},
        {"path": "tests/owl/DUL.owl", "format": "rdf-xml"},
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "somaads",
			"read-only": False
		}
	],
	"reasoner": [
        {
            "name": "ADReasoner",
            "type": "ADReasoner",
            "module": "/home/malineni/ROS_WS/knowrob/tutorials/ActionDesignatorReasoner.py",
			"data-backend": "mongodb",
        }
    ]
}
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)
instruction = input("Enter the instruction: ")  # cut the brocolli using the big sharp black knife on the desk
json_str = json.dumps(sample_dict)
query_temp = f'nlquery:nlquery("{instruction}", ?response)'
# nlquery:nlquery("cut the brocolli using the big sharp black knife on the desk", ?response)
print(instruction,"\n",query_temp)
phi1 = QueryParser.parse(f'{query_temp}')
phi1

[15:40:33.878] [info] Using backend `mongodb` with type `MongoDB`.
[15:40:33.878] [info] [mongodb] connected to mongodb://localhost:27017 (somaads.triples).
[15:40:33.885] [info] Using queryable backend with id 'mongodb'.
[15:40:33.885] [info] Using persistent backend with id 'mongodb'.
[15:40:33.971] [info] Using reasoner `ADReasoner` with type `ADReasoner`.
[15:40:33.971] [info] Using goal-driven reasoner with id 'ADReasoner'.
cut the apple 
 nlquery:nlquery("cut the apple", ?response)


nlquery:nlquery("cut the apple", ?response)

In [11]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()


=== Processing Designator String ===[13:53:23.106] [error] flask response text type is <class 'str'>
[13:53:23.106] [error] LLM response: ```
(an action
    (type fastening)
    (object (an object
              (type screw)
              (name "metal-screw")
              (properties (size "small")
                          (material "steel")
                          (thread-type "phillips")
                          (head-type "round"))))
    (tool (a tool
            (type wrench)
            (name "adjustable-wrench")
            (properties (size "medium")
                        (material "steel")
                        (grip "rubber")
                        (adjustability "high"))))
    (location (a location
                (type workstation)
                (name "workbench")
                (properties (surface-type "stable")
                            (height 0.9)
                            (accessibility "high"))))
    (goal (for-object (an object
                      

In [ ]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

In [ ]:
# phi2 = QueryParser.parse('nlquery:nlquery("get the locomotion ontology class description", ?response)')
# resultStream = kb.submitQuery(phi2, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
# resultQueue = resultStream.createQueue()
# # Get the result
# nextResult2 = resultQueue.pop_front()
# if isinstance(nextResult2, AnswerNo):
#     print("result is negative")
# else:
#     print("result is positive")

### PyCRAPOntology

In [ ]:
from src.pycrap.ontology_wrapper import OntologyWrapper

In [ ]:
ow = OntologyWrapper()

In [ ]:
ow.ontology

In [ ]:
ow.python_objects

In [ ]:
ow.file

In [ ]:
ow.classes()

In [ ]:
[cls for cls in ow.classes()]

In [ ]:
[ind for ind in ow.individuals()]

In [ ]:
ow.reason(save_to="inferred_SOMA.owl")

In [ ]:
ow.python_objects

### Extending SOMA - Manual

In [ ]:
from owlready2 import *
onto = get_ontology("SOMA.owl").load()
dul = get_ontology("DUL.owl").load()
for imp in onto.imported_ontologies:
    print(imp.base_iri)

In [ ]:
# get_namespace("http://www.ease-crc.org/ont/SOMA.owl")

In [ ]:
dul.base_iri

In [ ]:
# dul = onto.imported_ontologies[0]
# for cls in onto.classes():
#     if cls.name == "PhysicalPlace":
#         print(cls, cls.namespace)
# print(dul)
# DUL = get_ontology("http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#").load()

In [ ]:
# [cls for cls in DUL.classes()]
# [cls for cls in onto.classes()]

In [ ]:
dul.PhysicalObject

In [ ]:
with onto:
    class ForceMinimalExertion(onto.ForceAttribute):
        pass
    class ForceProprioceptionFeedback(onto.ForceAttribute):
        pass
    class ForceMinorResistance(onto.ForceAttribute):
        pass
    class ForceContactForce(onto.ForceAttribute):
        pass
    class ForceResistanceFromObject(onto.ForceAttribute):
        pass
    class ForceEquilibriumHold(onto.ForceAttribute):
        pass
    class ForceEquilibriumState(onto.ForceAttribute):
        pass
    class ForceGravityCompensation(onto.ForceAttribute):
        pass
    class ForcePreloading(onto.ForceAttribute):
        pass
    class ForceShearForce(onto.ForceAttribute):
        pass
    class ForceCompressiveForce(onto.ForceAttribute):
        pass
    class ForceBlockageEvent(onto.ForceAttribute):
        pass
    class ForceBreakageEvent(onto.ForceAttribute):
        pass
    class ForceModulationDuringCutting(onto.ForceAttribute):
        pass
    class ForceLoadReduction(onto.ForceAttribute):
        pass
    class ForceEquilibriumReset(onto.ForceAttribute):
        pass
    class ForceFrictionGrip(onto.StaticFrictionAttribute):
        pass
    class ForceFrictionResistance(onto.KineticFrictionAttribute):
        pass
    class ForceSeparationEvent(onto.NetForce):
        pass

ForceMinimalExertion.comment = "Minimal force exerted during free-space movement, e.g., moving a knife through the air before contact."
ForceMinimalExertion.label = "Minimal Exertion"
ForceMinimalExertion.isDefinedBy = onto.base_iri

ForceProprioceptionFeedback.comment = "Force sensed at joints or end-effectors, providing feedback during movement or grasping."
ForceProprioceptionFeedback.label = "Proprioception Feedback"
ForceProprioceptionFeedback.isDefinedBy = onto.base_iri

ForceMinorResistance.comment = "Small opposing force encountered, such as air resistance or workspace constraints, during movement."
ForceMinorResistance.label = "Minor Resistance"
ForceMinorResistance.isDefinedBy = onto.base_iri

ForceContactForce.comment = "Initial force applied when grasping an object, marking the onset of contact."
ForceContactForce.label = "Contact Force"
ForceContactForce.isDefinedBy = onto.base_iri

ForceResistanceFromObject.comment = "Force exerted back by an object due to its shape, affecting grasp efficiency."
ForceResistanceFromObject.label = "Resistance From Object"
ForceResistanceFromObject.isDefinedBy = onto.base_iri

ForceEquilibriumHold.comment = "Stable force maintained to hold an object securely during grasping."
ForceEquilibriumHold.label = "Equilibrium Hold"
ForceEquilibriumHold.isDefinedBy = onto.base_iri

ForceEquilibriumState.comment = "Minimal force state maintained before contact with an object."
ForceEquilibriumState.label = "Equilibrium State"
ForceEquilibriumState.isDefinedBy = onto.base_iri

ForceGravityCompensation.comment = "Force applied to hold an object, like a knife, stable against the effect of gravity."
ForceGravityCompensation.label = "Gravity Compensation"
ForceGravityCompensation.isDefinedBy = onto.base_iri

ForcePreloading.comment = "Slight downward force applied before initiating a cutting action."
ForcePreloading.label = "Preloading"
ForcePreloading.isDefinedBy = onto.base_iri

ForceShearForce.comment = "Lateral force exerted by a blade during cutting, causing shear stress on the object."
ForceShearForce.label = "Shear Force"
ForceShearForce.isDefinedBy = onto.base_iri

ForceCompressiveForce.comment = "Downward pressure exerted on an object, such as an apple, during cutting."
ForceCompressiveForce.label = "Compressive Force"
ForceCompressiveForce.isDefinedBy = onto.base_iri

ForceBlockageEvent.comment = "Initial resistance force encountered before a blade penetrates an object."
ForceBlockageEvent.label = "Blockage Event"
ForceBlockageEvent.isDefinedBy = onto.base_iri

ForceBreakageEvent.comment = "Force at which an object, like an apple, fractures as cutting completes."
ForceBreakageEvent.label = "Breakage Event"
ForceBreakageEvent.isDefinedBy = onto.base_iri

ForceModulationDuringCutting.comment = "Dynamic adjustment of pressure applied during the cutting process."
ForceModulationDuringCutting.label = "Modulation During Cutting"
ForceModulationDuringCutting.isDefinedBy = onto.base_iri

ForceLoadReduction.comment = "Gradual reduction in force applied after completing a cutting action."
ForceLoadReduction.label = "Load Reduction"
ForceLoadReduction.isDefinedBy = onto.base_iri

ForceEquilibriumReset.comment = "Force state where the arm stabilizes after completing a task."
ForceEquilibriumReset.label = "Equilibrium Reset"
ForceEquilibriumReset.isDefinedBy = onto.base_iri

ForceFrictionGrip.comment = "Static friction force that prevents slipping during grasping of an object."
ForceFrictionGrip.label = "Friction Grip"
ForceFrictionGrip.isDefinedBy = onto.base_iri

ForceFrictionResistance.comment = "Kinetic friction force encountered by a blade from an object's fibers during cutting."
ForceFrictionResistance.label = "Friction Resistance"
ForceFrictionResistance.isDefinedBy = onto.base_iri

ForceSeparationEvent.comment = "Accumulated force causing apple halves to move apart under gravity after cutting."
ForceSeparationEvent.label = "Separation Event"
ForceSeparationEvent.isDefinedBy = onto.base_iri

In [ ]:
with onto:
    class exerts_force(dul.associatedWith):
        domain = [dul.PhysicalObject]
        range = [onto.ForceAttribute]
        transitive = True

    class is_exerted_by(dul.associatedWith):
        domain = [onto.ForceAttribute]
        range = [dul.PhysicalObject]
        transitive = True
        inverse_property = exerts_force

    class resists_force(dul.associatedWith):
        domain = [dul.PhysicalObject]
        range = [onto.ForceAttribute]
        transitive = True

    class is_resisted_by(dul.associatedWith):
        domain = [onto.ForceAttribute]
        range = [dul.PhysicalObject]
        transitive = True
        inverse_property = resists_force

    class occurs_during(dul.associatedWith):
        domain = [onto.ForceAttribute]
        range = [dul.Action]
        transitive = False

    class applied_to(dul.associatedWith):
        domain = [dul.Action]
        range = [onto.ForceAttribute]
        transitive = False

exerts_force.comment = "A relation indicating that a physical object exerts a specific force attribute, e.g., 'a knife exerts shear force on an apple'."

is_exerted_by.comment = "A relation indicating that a force attribute is exerted by a physical object, e.g., 'shear force is exerted by a knife'."

resists_force.comment = "A relation indicating that a physical object resists a specific force attribute, e.g., 'an apple resists friction force from a knife'."

is_resisted_by.comment = "A relation indicating that a force attribute is resisted by a physical object, e.g., 'friction force is resisted by an apple'."

occurs_during.comment = "A relation indicating that a force attribute occurs during a specific action, e.g., 'compressive force occurs during cutting'."

applied_to.comment = "A relation indicating that a force attribute is applied to a physical object, e.g., 'compressive force is applied to an apple'."

In [ ]:
onto.save(file="extended_ontology.owl", format="rdfxml")

### Mappings

In [ ]:
action_designator = {
    "type": "place",
    "object": {"type": "mug", "color": "blue"},
    "location": {"on": "table"}
}

In [ ]:
# with onto:
#     sync_reasoner()

In [ ]:
# soma = onto.get_namespace("http://www.ease-crc.org/ont/SOMA.owl#")
# dul = onto.get_namespace("http://www.ontologydesignpatterns.org/ont/dul/DUL.owl#")
dul.PhysicalObject

In [ ]:
def instantiate_designator(designator):
    with onto:
        # Create the action instance
        action = onto.Placing("placing_1")  # Unique ID for the action
        # action.is_a.append(onto.PhysicalTask)  # Ensure type hierarchy
        # action.is_a.append(owlready2.Thing)

        blue = onto.Color("blue")

        # Create the object instance (mug)
        mug = dul.PhysicalObject("mug_1")
        # mug.hasColor.append(blue)  # Add color property

        # Create the location instance (table)
        table = onto.Location("table_1")

        # # Link action to object and location
        # mug.isParticipantIn.append(action)
        # action.hasParticipant.append(mug)
        # action.hasParticipant.append(table)
        # # table.isLocationOf.append(action)
        # table.isLocationOf.append(mug)

    return action, mug, table

In [ ]:
instantiate_designator(action_designator)

In [ ]:
onto.save(file="extended_ontology.owl", format="rdfxml")

In [ ]:
import json
from knowrob import *
InitKnowRob()

In [ ]:
# Sample dictionary to be converted to JSON
sample_dict = {
	"logging": {
		"console-sink": {"level": "debug"},
		"file-sink": {"level": "debug"}
	},
	"semantic-web": {
		"prefixes": [
			{"alias": "soma", "uri": "http://www.ease-crc.org/ont/SOMA.owl"},
            {"alias": "dul", "uri": "http://www.ontologydesignpatterns.org/ont/dul/DUL.owl"},
            {"alias": "nlquery", "uri": "http://knowrob.org/kb/nlquery"}
		]
	},
	"data-sources": [
		{"path": "tutorials/extended_ontology.owl", "format": "rdf-xml"}
	],
	"data-backends": [
		{
			"type": "MongoDB",
			"name": "mongodb",
			"host": "localhost",
			"port": 27017,
			"db": "xSOMA",
			"read-only": False
		}
	],
	"reasoner": [
        {
            "name": "ADReasoner",
            "type": "ADReasoner",
            "module": "/home/malineni/ROS_WS/knowrob/tutorials/ActionDesignatorReasoner.py",
			"data-backend": "mongodb",
        }
    ]
}

In [ ]:
# Convert the dictionary to a JSON string
json_str = json.dumps(sample_dict)
# Initialize the KnowledgeBase with the PropertyTree
kb = KnowledgeBase(json_str)

In [ ]:
phi1 = QueryParser.parse("soma:hasColor(soma:'mug_1', ?y)")

In [ ]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()

In [ ]:
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

In [ ]:
# phi1 = QueryParser.parse("dul:isParticipantIn(soma:'mug_1', ?y)")
phi1 = QueryParser.parse("dul:hasParticipant(soma:'placing_1', ?y)")

In [ ]:
resultStream = kb.submitQuery(phi1, QueryContext(QueryFlag.QUERY_FLAG_ALL_SOLUTIONS))
resultQueue = resultStream.createQueue()
# Get the result
nextResult1 = resultQueue.pop_front()
if isinstance(nextResult1, AnswerYes):
    for substitution in nextResult1.substitution():
        variable = substitution[1]
        term = substitution[2]
        print(str(variable) + " : " + str(term))

#### Ex2


### Restrictions by ClassName (incl. inherited restrictions)

In [ ]:
from owlready2 import *
import inspect

def get_class_restrictions(ontology, class_name):
    """
    Retrieve and print all restrictions for a given class in an ontology

    Parameters:
    ontology: The loaded ontology
    class_name: The name of the class to examine
    """
    # Get the class object
    try:
        target_class = getattr(ontology, class_name)
    except AttributeError:
        print(f"Class {class_name} not found in the ontology")
        return

    print(f"Restrictions for class: {class_name}")
    print("-" * 50)

    # Get all subclass restrictions (these are the restrictions applied to the class)
    for restriction in target_class.is_a:
        # Check if it's a restriction or a class
        if not hasattr(restriction, "__class__"):
            continue

        # Handle different types of restrictions
        if "Restriction" in restriction.__class__.__name__:
            # It's a restriction, get its components
            property_name = restriction.property.name if hasattr(restriction.property, "name") else str(restriction.property)
            restriction_type = restriction.__class__.__name__

            # Different handling based on restriction type
            if hasattr(restriction, "value"):
                value = restriction.value
                value_name = value.name if hasattr(value, "name") else str(value)
                print(f"Property: {property_name}, Type: hasValue, Value: {value_name}")

            elif hasattr(restriction, "cardinality"):
                card = restriction.cardinality
                value_type = restriction.value_type.name if hasattr(restriction.value_type, "name") else str(restriction.value_type)
                print(f"Property: {property_name}, Type: exactly, Cardinality: {card}, On: {value_type}")

            elif hasattr(restriction, "min_cardinality"):
                card = restriction.min_cardinality
                value_type = restriction.value_type.name if hasattr(restriction.value_type, "name") else str(restriction.value_type)
                print(f"Property: {property_name}, Type: min, Cardinality: {card}, On: {value_type}")

            elif hasattr(restriction, "max_cardinality"):
                card = restriction.max_cardinality
                value_type = restriction.value_type.name if hasattr(restriction.value_type, "name") else str(restriction.value_type)
                print(f"Property: {property_name}, Type: max, Cardinality: {card}, On: {value_type}")

            elif hasattr(restriction, "value_type"):
                value_type = restriction.value_type.name if hasattr(restriction.value_type, "name") else str(restriction.value_type)
                if "some" in restriction.__class__.__name__.lower():
                    print(f"Property: {property_name}, Type: some, On: {value_type}")
                elif "only" in restriction.__class__.__name__.lower():
                    print(f"Property: {property_name}, Type: only, On: {value_type}")
                else:
                    print(f"Property: {property_name}, Type: {restriction_type}, On: {value_type}")
            else:
                print(f"Property: {property_name}, Type: {restriction_type}, Value: [complex]")
        else:
            # It's a parent class
            parent_name = restriction.name if hasattr(restriction, "name") else str(restriction)
            print(f"Parent class: {parent_name}")

    print("-" * 50)
    print("Inherited restrictions:")
    print("-" * 50)

    # Get all ancestor classes to check their restrictions too
    ancestors = list(target_class.ancestors())
    for ancestor in ancestors:
        if ancestor == target_class:
            continue

        ancestor_name = ancestor.name if hasattr(ancestor, "name") else str(ancestor)
        if not ancestor_name or ancestor_name == "Thing":
            continue

        print(f"From ancestor: {ancestor_name}")

        for restriction in ancestor.is_a:
            if not hasattr(restriction, "__class__"):
                continue

            if "Restriction" in restriction.__class__.__name__:
                property_name = restriction.property.name if hasattr(restriction.property, "name") else str(restriction.property)
                restriction_type = restriction.__class__.__name__

                if hasattr(restriction, "value"):
                    value = restriction.value
                    value_name = value.name if hasattr(value, "name") else str(value)
                    print(f"  Property: {property_name}, Type: hasValue, Value: {value_name}")

                elif hasattr(restriction, "cardinality"):
                    card = restriction.cardinality
                    value_type = restriction.value_type.name if hasattr(restriction.value_type, "name") else str(restriction.value_type)
                    print(f"  Property: {property_name}, Type: exactly, Cardinality: {card}, On: {value_type}")

                elif hasattr(restriction, "min_cardinality"):
                    card = restriction.min_cardinality
                    value_type = restriction.value_type.name if hasattr(restriction.value_type, "name") else str(restriction.value_type)
                    print(f"  Property: {property_name}, Type: min, Cardinality: {card}, On: {value_type}")

                elif hasattr(restriction, "max_cardinality"):
                    card = restriction.max_cardinality
                    value_type = restriction.value_type.name if hasattr(restriction.value_type, "name") else str(restriction.value_type)
                    print(f"  Property: {property_name}, Type: max, Cardinality: {card}, On: {value_type}")

                elif hasattr(restriction, "value_type"):
                    value_type = restriction.value_type.name if hasattr(restriction.value_type, "name") else str(restriction.value_type)
                    if "some" in restriction.__class__.__name__.lower():
                        print(f"  Property: {property_name}, Type: some, On: {value_type}")
                    elif "only" in restriction.__class__.__name__.lower():
                        print(f"  Property: {property_name}, Type: only, On: {value_type}")
                    else:
                        print(f"  Property: {property_name}, Type: {restriction_type}, On: {value_type}")
                else:
                    print(f"  Property: {property_name}, Type: {restriction_type}, Value: [complex]")

In [ ]:
world = World()
SOMA = world.get_ontology("SOMA.owl").load()

# Get restrictions for DesignedContainer
get_class_restrictions(SOMA, "DesignedContainer")

In [ ]:
import sexpdata
import sys
from typing import Any

# Input designator string
designator_string = """
(an action
    (type cutting)
    (object (an object
              (type apple)
              (name "apple")
              (properties (size "medium")
                          (texture "bumpy")
                          (color "green")))))
"""

# Conversion function
def convert_to_tuples(data: Any) -> Any:
    """
    Recursively converts parsed sexpdata (lists, Symbols, strings, numbers)
    into nested tuples containing only basic Python types (tuples, strings, numbers).
    """
    if isinstance(data, list):
        return tuple(convert_to_tuples(item) for item in data)
    elif isinstance(data, sexpdata.Symbol):
        return data.value()
    elif isinstance(data, (str, int, float, bool, type(None))):
         return data
    else:
        # Fallback for unexpected types
        return str(data)

# Parsing, Conversion, and Final Output
try:
    parsed_data = sexpdata.loads(designator_string)
    designator_tuple = convert_to_tuples(parsed_data)
    # --- ONLY print the final tuple representation ---
    print(designator_tuple)
    # --- End final output ---
except sexpdata.SexpException as e:
    print(f"Error parsing designator: {e}", file=sys.stderr)
except Exception as e:
    print(f"An unexpected error occurred: {e}", file=sys.stderr)

In [ ]:
designator_tuple